# Seoul Rescue Activity Patterns & Dispatch Delay (2021)

This notebook is a **portfolio-style walkthrough** of a small public-data analysis project.

**What you will see**
- loading and validating a real-world CSV dataset
- parsing timestamps and creating a “dispatch delay” feature
- basic data-quality checks
- descriptive analysis and clear visualisations

**Data source (dataset series)**  
Korea National Fire Agency (NFA): “Rescue Activity Status” (Public Data Portal)  
https://www.data.go.kr/data/15062386/fileData.do

> The portal describes the dataset as a record of rescue activities including report time, dispatch time, incident location, and incident type.  
> The licence on the portal page states **no restrictions** (“이용허락범위 제한 없음”).

---

## 0) Setup


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Local helper functions (kept in src/ for reuse)
import sys
sys.path.append(str(Path.cwd().parents[0] / "src"))

from utils import (
    COL_CATEGORY,
    COL_CITY,
    COL_DISTRICT,
    add_time_features,
    apply_basic_qc,
    load_seoul_data,
    translate_categories,
)

project_root = Path.cwd().parents[0]
data_path = project_root / "data" / "fire_rescue_seoul_2021.csv"
fig_dir = project_root / "reports" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

data_path


## 1) Load data

The file in this repository is a **Seoul-only subset** of the annual dataset, saved in UTF‑8 for convenience.


In [ ]:
df = load_seoul_data(data_path)

print("Shape:", df.shape)
df.head()

### Quick schema check

Before doing any analysis, it’s good practice to confirm:
- expected columns exist
- there are no obvious missing values in key fields


In [ ]:
expected_cols = [
    "번호",
    "신고년월일", "신고시각",
    "출동년월일", "출동시각",
    "발생장소_시", "발생장소_구", "발생장소_동",
    "사고원인", "사고원인코드명_사고종별",
]

missing = [c for c in expected_cols if c not in df.columns]
print("Missing columns:", missing)

df[expected_cols].isna().mean().sort_values(ascending=False).head(10)


## 2) Feature engineering: timestamps and dispatch delay

We combine:
- `신고년월일` + `신고시각` → `report_dt`
- `출동년월일` + `출동시각` → `dispatch_dt`

Then compute a simple delay metric:

\[
\text{dispatch\_delay\_min} = \text{dispatch\_dt} - \text{report\_dt}
\]

This is a **report → dispatch** delay (not an on-scene arrival time).


In [ ]:
df_feat = add_time_features(df)

df_feat[["report_dt", "dispatch_dt", "dispatch_delay_min", "month", "hour", "dow"]].head()


## 3) Data quality checks

Real operational data often contains quirks (manual entry errors, system timestamps, etc.).  
Here we apply **minimal QC**:

- exclude negative delays
- exclude very large delays (> 120 minutes) when summarising “typical” dispatch delay

We keep the excluded records separately so we can report how much data was removed.


In [ ]:
clean, excluded = apply_basic_qc(df_feat)

print("Total records:", len(df_feat))
print("Clean records:", len(clean))
print("Excluded records:", len(excluded))
print("Excluded share (%):", round(100 * len(excluded) / len(df_feat), 3))

clean["dispatch_delay_min"].describe(percentiles=[0.5, 0.9, 0.95, 0.99])


## 4) Descriptive analysis & visualisations

### 4.1 Monthly incident volume

In [ ]:
monthly = clean.groupby(clean["report_dt"].dt.to_period("M")).size().sort_index()

plt.figure(figsize=(8, 4.5))
plt.plot(monthly.index.astype(str), monthly.values, marker="o")
plt.xticks(rotation=45, ha="right")
plt.title("Seoul Rescue Activity Volume by Month (2021)")
plt.xlabel("Month")
plt.ylabel("Number of records")
plt.tight_layout()

out = fig_dir / "monthly_volume.png"
plt.savefig(out, dpi=200)
out


### 4.2 Most common incident categories

The category labels are in Korean (administrative coding).  
For readability in the charts, we translate the most frequent ones to English.


In [ ]:
top_cat = clean[COL_CATEGORY].value_counts().head(12)
top_cat.index = translate_categories(top_cat.index.to_series())
top_cat = top_cat.sort_values()

plt.figure(figsize=(8, 5.5))
plt.barh(top_cat.index, top_cat.values)
plt.title("Top Incident Categories in Seoul (2021)")
plt.xlabel("Count")
plt.tight_layout()

out = fig_dir / "top_categories_eng.png"
plt.savefig(out, dpi=200)
out


### 4.3 Dispatch delay distribution

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.hist(clean["dispatch_delay_min"].clip(upper=20), bins=40)
plt.title("Dispatch Delay Distribution (clipped at 20 minutes)")
plt.xlabel("Minutes from report to dispatch")
plt.ylabel("Frequency")
plt.tight_layout()

out = fig_dir / "dispatch_delay_hist.png"
plt.savefig(out, dpi=200)
out


### 4.4 Incident timing patterns (day-of-week × hour)

This heatmap helps spot operational patterns such as evening peaks or weekend differences.


In [ ]:
pivot = clean.pivot_table(
    index=clean["report_dt"].dt.day_name(),
    columns=clean["report_dt"].dt.hour,
    values="번호",
    aggfunc="count",
    fill_value=0,
)
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
pivot = pivot.reindex(order)

plt.figure(figsize=(11, 4.5))
plt.imshow(pivot.values, aspect="auto")
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xticks(range(0, 24, 2), list(range(0, 24, 2)))
plt.title("Incident Volume Heatmap (Day of Week x Hour)")
plt.xlabel("Hour of day")
plt.ylabel("Day of week")
plt.colorbar(label="Count")
plt.tight_layout()

out = fig_dir / "dow_hour_heatmap.png"
plt.savefig(out, dpi=200)
out


### 4.5 Dispatch delay by category (top 8)

Boxplots are useful to compare typical delays across categories, while being robust to outliers.


In [ ]:
top_for_box = clean[COL_CATEGORY].value_counts().head(8).index
box_data = [clean.loc[clean[COL_CATEGORY] == c, "dispatch_delay_min"] for c in top_for_box]
labels = list(translate_categories(top_for_box.to_series()))

plt.figure(figsize=(10, 5))
plt.boxplot(box_data, labels=labels, showfliers=False)
plt.xticks(rotation=45, ha="right")
plt.title("Dispatch Delay by Category (Top 8, outliers hidden)")
plt.ylabel("Minutes")
plt.tight_layout()

out = fig_dir / "delay_by_category_box_eng.png"
plt.savefig(out, dpi=200)
out


## 5) Short interpretation (non-causal)

This portfolio analysis is intentionally descriptive:

- It shows how to turn operational logs into analysis-ready features (datetimes, delays).
- It highlights basic measurement issues (negative timestamps, extreme outliers).
- It provides readable summaries and visuals.

**Next steps (if extending this project)**
- enrich with weather, neighbourhood characteristics, or staffing data
- define a clearer outcome (e.g., on-scene arrival) if available
- use quasi-experimental methods only once the data supports credible identification
